<a href="https://colab.research.google.com/github/bautistabc/AI_llama/blob/master/Copia_de_Hands_On_Pipeline_Completo_(Soluci%C3%B3n).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PIPELINE COMPLETO: DE DATOS A MODELO DESPLEGADO**

Este Colab conecta en un solo flujo lo visto en los tres Temas anteriores (inferencia básica, RAG y fine-tuning con LoRa) para construir un asistente que recupera contexto propio y responde con un modelo ya ajustado a un tono específico. Es la base directa para el reto del Hackathon 1.

**Nota de alcance:** "Desplegar" aquí significa guardar el modelo ajustado de forma reutilizable y envolverlo en una función lista para usar, no se levanta un servidor ni un endpoint público.

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [ ]:
# Instalar librerías e iniciar sesión en Hugging Face con el token desde Colab Secrets

!pip install transformers peft accelerate trl sentence-transformers --quiet

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

login(token=userdata.get('HF_TOKEN'))
print("Sesión de Hugging Face iniciada correctamente.")

Sesión de Hugging Face iniciada correctamente.


## **PASO 1: LA BASE DE CONOCIMIENTO (RAG)**

Indexamos una base de conocimiento propia con `sentence-transformers` para poder recuperar el fragmento más relevante antes de responder, esto es lo que evita que el asistente alucine sobre políticas que no conoce.

In [ ]:
# Generar los embeddings de la base de conocimiento (política de devoluciones)

from sentence_transformers import SentenceTransformer
import numpy as np

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en "
    "su empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo "
    "de envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


In [ ]:
# Definir la función de recuperación (RAG)

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

buscar_fragmento("¿Puedo devolver algo que compré en oferta?")

'Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.'

## **PASO 2: AJUSTAR EL TONO DEL MODELO (FINE-TUNING CON LoRA)**

Cargamos un modelo ligero basado en Llama y lo ajustamos con LoRA para que responda siempre en el mismo tono breve y directo, la configuración de esta celda ya fue validada en el tema 3.

In [ ]:
# Cargar el modelo base y su tokenizer

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [ ]:
def generar_respuesta(modelo_a_usar, pregunta, max_new_tokens=60):
    mensajes = [{"role": "user", "content": pregunta}]
    prompt_formateado = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(prompt_formateado, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    return tokenizer.decode(tokens_nuevos, skip_special_tokens=True).strip()

prompt_prueba = "¿Puedo cambiar mi pedido después de pagarlo?"

In [ ]:
# Definir el dataset de ejemplos y convertirlo en Dataset

from datasets import Dataset

def formatear_ejemplo(pregunta, respuesta):
    mensajes = [
        {"role": "user", "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    return tokenizer.apply_chat_template(mensajes, tokenize=False)

pares = [
    ("¿Puedo cambiar mi pedido después de pagarlo?", "Sí, puedes solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com."),
    ("¿Cuánto tarda el reembolso?", "El reembolso se refleja en un plazo de 5 a 7 días hábiles."),
    ("¿Tienen envío el mismo día?", "Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."),
    ("¿Puedo pagar en el momento de la entrega?", "Sí, aceptamos pago contra entrega en efectivo o tarjeta."),
    ("¿Cómo rastreo mi paquete?", "Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."),
]

ejemplos = [{"texto": formatear_ejemplo(p, r)} for p, r in pares]
dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

In [ ]:
# Probar el modelo base con el prompt de prueba antes de ajustarlo

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

Sí, puedes cambiar tu pedido a cualquier momento después de haber pagadolo. Si necesitas cambiar el pedido, puede contactar con el servicio de soporte técnico de la tienda para obtener más información sobre cómo hacerlo.

Si


In [ ]:
# Configurar LoRA (semilla fija para un resultado reproducible en la grabación)

!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# Entrenar con LoRA

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados_pipeline",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = trainer.state.log_history[-2]['loss'] # último paso del entrenamiento
#perdida_final = resultado_entrenamiento.training_loss # promedio del entrenamiento
print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '2.63', 'grad_norm': '2.816', 'learning_rate': '0.0002', 'entropy': '1.507', 'num_tokens': '279', 'mean_token_accuracy': '0.5839', 'epoch': '1'}
{'loss': '2.571', 'grad_norm': '2.676', 'learning_rate': '0.0001933', 'entropy': '1.51', 'num_tokens': '558', 'mean_token_accuracy': '0.5839', 'epoch': '2'}
{'loss': '2.484', 'grad_norm': '2.68', 'learning_rate': '0.0001867', 'entropy': '1.506', 'num_tokens': '837', 'mean_token_accuracy': '0.5839', 'epoch': '3'}
{'loss': '2.387', 'grad_norm': '2.751', 'learning_rate': '0.00018', 'entropy': '1.509', 'num_tokens': '1116', 'mean_token_accuracy': '0.5912', 'epoch': '4'}
{'loss': '2.277', 'grad_norm': '2.866', 'learning_rate': '0.0001733', 'entropy': '1.51', 'num_tokens': '1395', 'mean_token_accuracy': '0.5949', 'epoch': '5'}
{'loss': '2.169', 'grad_norm': '3.031', 'learning_rate': '0.0001667', 'entropy': '1.517', 'num_tokens': '1674', 'mean_token_accuracy': '0.5985', 'epoch': '6'}
{'loss': '2.053', 'grad_norm': '3.269', 'learning_rate': '

## **PASO 3: DESPLEGAR EL MODELO AJUSTADO**

"Desplegar" en este contexto significa guardar el adaptador LoRA de forma reutilizable, no levantar un servidor. Con `save_pretrained` queda listo para volver a cargarlo en cualquier notebook sin repetir el entrenamiento; `push_to_hub` es opcional si quieres tenerlo disponible en tu cuenta de Hugging Face.

In [ ]:
# Guardar el adaptador LoRA localmente

modelo_lora.save_pretrained("/content/modelo_ajustado_lora")
print("Adaptador LoRA guardado en /content/modelo_ajustado_lora")

# Opcional: subir el adaptador a tu cuenta de Hugging Face para reutilizarlo fuera de esta sesión
# modelo_lora.push_to_hub("tu-usuario/tinyllama-atencion-clientes-lora")

Adaptador LoRA guardado en /content/modelo_ajustado_lora


## **PASO 4: EL PIPELINE COMPLETO — RAG + MODELO AJUSTADO**

Con la base de conocimiento indexada y el modelo ya ajustado, conectamos ambas piezas en una sola función: recupera el fragmento relevante, se lo entrega al modelo ajustado junto con la pregunta, y genera la respuesta final. Este es el mismo patrón que se espera construir en el Hackathon 1.

***Nota:** TinyLlama aprendió un tono específico con solo 5 ejemplos, suficiente para demostrar que LoRA funciona, pero no para generalizar a una instrucción nueva con contexto inyectado, como la que pide este pipeline con RAG.*

*Por lo que, un modelo real desplegado sí tendría esa capacidad, usando aquí Groq (GPT-OSS-20B) en lugar del modelo ajustado con LoRA como muestra de ese comportamiento.*

In [ ]:
# Función del asistente: RAG (recuperación) + modelo desplegado (generación)

!pip install groq -q

from groq import Groq
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def asistente(pregunta):
    fragmento = buscar_fragmento(pregunta)
    prompt = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

for pregunta in [
    "¿Puedo devolver algo que compré en oferta?",
    "¿Cuánto tarda mi reembolso?",
    "¿El envío internacional tiene devolución gratis?",
]:
    print(f"Pregunta: {pregunta}")
    print(f"Respuesta: {asistente(pregunta)}\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.5 MB/s eta 0:00:00
Pregunta: ¿Puedo devolver algo que compré en oferta?
Respuesta: No, los productos en oferta no son elegibles para devolución, solo para cambio de talla.

Pregunta: ¿Cuánto tarda mi reembolso?
Respuesta: Lo siento, la política no cubre esa información.

Pregunta: ¿El envío internacional tiene devolución gratis?
Respuesta: No, los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de envío de regreso.

